In [ ]:
import yaml
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import shap
import pickle
import xgboost as xgb
import os

with open('../config.yaml', 'r', encoding='utf-8') as f:
    cfg = yaml.safe_load(f)

CLASS_NAMES = cfg['classes']['names']
print("Classes:", CLASS_NAMES)


In [ ]:
# Load raw data for visualization
raw_df = pd.read_pickle('../' + cfg['data']['raw_path'])

raw_df['failureType'] = raw_df['failureType'].apply(lambda x: x[0][0] if isinstance(x, np.ndarray) and len(x)>0 and len(x[0])>0 else 'none')
raw_df['failureType'] = raw_df['failureType'].replace({'Near-full': 'Normal'})
raw_df = raw_df[raw_df['failureType'].isin(CLASS_NAMES)]

fig, axes = plt.subplots(8, 5, figsize=(15, 20))
fig.subplots_adjust(hspace=0.3, wspace=0.1)

for i, cls in enumerate(CLASS_NAMES):
    cls_samples = raw_df[raw_df['failureType'] == cls].head(5)
    for j, (_, row) in enumerate(cls_samples.iterrows()):
        ax = axes[i, j]
        ax.imshow(row['waferMap'], cmap='inferno')
        ax.set_xticks([])
        ax.set_yticks([])
        if j == 0:
            ax.set_ylabel(cls, fontsize=12)

plt.suptitle('Representative Wafer Maps (8 Classes × 5 Samples)', fontsize=16, y=0.92)
os.makedirs('../results/figures', exist_ok=True)
plt.savefig('../results/figures/sample_wafers_grid.png', bbox_inches='tight')
plt.show()


In [ ]:
# SHAP 결과 로드 + 클래스별 Top-5 피처 테이블
# Hybrid XGBoost model 직접 학습하여 shap_values 계산

features_df = pd.read_pickle('../' + cfg['data']['features_v2'])
cnn_emb = np.load('../' + cfg['data']['cnn_embeddings'])

# Manual features 
X_manual = features_df.drop(columns=['failureType'])
# CNN features
X_cnn = pd.DataFrame(cnn_emb, columns=[f'cnn_{i}' for i in range(cnn_emb.shape[1])])

# Combine Hybrid Features
X_hybrid = pd.concat([X_manual.reset_index(drop=True), X_cnn.reset_index(drop=True)], axis=1)
y = features_df['failureType'].reset_index(drop=True)

# Train a quick Hybrid model to get SHAP values
model = xgb.XGBClassifier(tree_method='hist', n_estimators=50, random_state=42)

# Label Encoding
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
y_encoded = le.fit_transform(y)
model.fit(X_hybrid, y_encoded)

# Compute SHAP
explainer = shap.TreeExplainer(model)
# Sample 1000 for fast SHAP computation in notebook
X_sample = shap.sample(X_hybrid, 1000, random_state=42)
shap_values = explainer.shap_values(X_sample)

# Display Top-5 features for each class
for i, cls in enumerate(le.classes_):
    print(f"\n[{cls}] Top 5 Features:")
    if isinstance(shap_values, list):
        vals = np.abs(shap_values[i]).mean(0)
    else:
        if hasattr(shap_values, 'values'):
            vals = np.abs(shap_values.values[:, :, i]).mean(0)
        else:
            vals = np.abs(shap_values[:, :, i]).mean(0)
            
    feature_importance = pd.DataFrame(list(zip(X_hybrid.columns, vals)), columns=['Feature', 'SHAP_importance'])
    feature_importance.sort_values(by=['SHAP_importance'], ascending=False, inplace=True)
    print(feature_importance.head(5).to_string(index=False))


## 패턴-공정-액션 매핑

| 패턴 | Top SHAP 피처 | 의심 공정 | 실무 액션 |
|------|--------------|----------|----------|
| Edge-Ring | edge_sector_std, radial_cdf_7 | CMP 패드 마모 / CVD 균일도 저하 | 패드 교체 주기 단축, 가장자리 레시피 점검 |
| Center | radial_cdf_0~2, cx/cy | CVD/스퍼터링 중앙 가스 과잉 | 샤워헤드 노즐 청소, 가스 비율 점검 |
| Donut | radial_cdf_3~5 | RTP/Annealing 히터 존 불균일 | 히터 존별 온도 프로파일 점검 |
| Scratch | compactness, linearity_pca | 이송 로봇 미끄러짐 | 이송 경로 로그, radon_max_angle 방향 특정 |
| Local | morans_i, n_clusters, lot_consistency | 챔버 파티클 오염 | lot_consistency로 일시적/구조적 판단 |
| Random | defect_ratio, morans_i | 공정 전반 불안정 | Cpk 모니터링, SPC 전체 지표 확인 |
| Edge-Loc | cx, cy, edge_sector_0/5 | 척(Chuck) 특정 부위 손상 | cx/cy 치우침 방향으로 척 핀 특정 |


In [ ]:
# 3방향 성능 비교 테이블
metrics = pd.read_csv('../results/metrics.csv')
print(metrics.to_string(index=False))


## Hybrid 선택 근거

- **수치 피처 20개**: 밀도·반경·클러스터 구조 포착
- **CNN 128차원**: 공간 텍스처·선형성·경계 형태 포착
- **Hybrid concat**: 두 표현을 동시에 활용 → macro F1 0.870
